# Packages & Libraries

In [1]:
from statsforecast import StatsForecast
from mlforecast import MLForecast
from mlforecast.target_transforms import Differences
from mlforecast.utils import PredictionIntervals
from window_ops.expanding import expanding_mean
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import RandomForestRegressor
from utilsforecast.plotting import plot_series

import mlflow
import mlforecast.flavor

import pandas as pd
import numpy as np
import requests
import json
import os
import datetime
from statistics import mean

/Users/aizhanm/Desktop/forecasting_service/.venv_frcst/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Data

In [2]:
DATA_TRAIN_FILE_PATH = "../data/train.csv"
PRODUCT_NAME = "BEVERAGES"
STORE_NUMBER = 1

In [3]:
ds = pd.read_csv(DATA_TRAIN_FILE_PATH)
ds = ds[(ds["family"] == PRODUCT_NAME) & (ds["store_nbr"] == STORE_NUMBER)]
ds["unique_id"] = 1  # ds["store_nbr"].astype(str) + "_" + ds["family"]
ds = ds[["date", "sales", "unique_id"]]
ds["date"] = pd.to_datetime(ds["date"], format="%Y-%m-%d")
ds = ds.sort_values("date")
ds = ds.rename(columns={"date": "ds", "sales": "y"})

In [4]:
fig = StatsForecast.plot(ds, engine="plotly")
fig.show()

In [5]:
ds["ds"] = pd.to_datetime(ds["ds"])

g = ds.sort_values("ds")

missing = pd.date_range(
    g["ds"].min(),
    g["ds"].max(),
    freq="D"
).difference(g["ds"])

missing = pd.DataFrame({
    "ds": pd.to_datetime(missing),
    "y": [0]*len(missing),
    "unique_id": [1]*len(missing)
})

ds = pd.concat([ds, missing], ignore_index=True).sort_values("ds")

In [6]:
train_size = int(len(ds) * 0.8)
train, test = ds.iloc[:train_size], ds.iloc[train_size:]

# Model

In [10]:
ml_models = {
    "lightGBM": LGBMRegressor(n_estimators=500, verbosity=-1),
    "xgboost": XGBRegressor(),
    "linear_regression": LinearRegression(),
    "lasso": Lasso(),
    "ridge": Ridge()
}

In [11]:
params = {
    "freq": "D",
    "lags": list(range(1, 56)),
    "date_features": ["year", "month", "day"]
}

In [12]:
mlf = MLForecast(
    models=ml_models,
    freq=params["freq"],
    lags=params["lags"],
    date_features=params["date_features"]
)

mlf.fit(train)

MLForecast(models=[lightGBM, xgboost, linear_regression, lasso, ridge], freq=D, lag_features=['lag1', 'lag2', 'lag3', 'lag4', 'lag5', 'lag6', 'lag7', 'lag8', 'lag9', 'lag10', 'lag11', 'lag12', 'lag13', 'lag14', 'lag15', 'lag16', 'lag17', 'lag18', 'lag19', 'lag20', 'lag21', 'lag22', 'lag23', 'lag24', 'lag25', 'lag26', 'lag27', 'lag28', 'lag29', 'lag30', 'lag31', 'lag32', 'lag33', 'lag34', 'lag35', 'lag36', 'lag37', 'lag38', 'lag39', 'lag40', 'lag41', 'lag42', 'lag43', 'lag44', 'lag45', 'lag46', 'lag47', 'lag48', 'lag49', 'lag50', 'lag51', 'lag52', 'lag53', 'lag54', 'lag55'], date_features=['year', 'month', 'day'], num_threads=1)

# Backtesting

In [14]:
#
partitions = 10 # size of partitions
step_size = 24 # window shift size
h = 72 # size of testing partitions

n_windows = 5
method = "conformal_distribution"
pi = PredictionIntervals(h=h, n_windows=n_windows, method=method)
levels = [95]

In [15]:
bkt_df = mlf.cross_validation(
    df = train,
    h = h,
    step_size=step_size,
    n_windows=partitions,
    prediction_intervals=pi,
    level=levels
)

In [16]:
bkt_df.head()

,unique_id,ds,cutoff,y,lightGBM,xgboost,linear_regression,lasso,ridge,lightGBM-lo-95,lightGBM-hi-95,xgboost-lo-95,xgboost-hi-95,linear_regression-lo-95,linear_regression-hi-95,lasso-lo-95,lasso-hi-95,ridge-lo-95,ridge-hi-95
0,1,2015-11-29,2015-11-28,1070.0,1042.754166,1013.568237,1197.485471,1195.065379,1197.392599,256.816119,1828.692212,38.063370,1989.073105,506.484125,1888.486817,493.410055,1896.720704,504.565430,1890.219769
1,1,2015-11-30,2015-11-28,2342.0,2041.674510,1981.151245,1994.031081,1990.449698,1993.892027,1120.465351,2962.883669,1315.927103,2646.375388,1176.963825,2811.098336,1163.937867,2816.961528,1175.223615,2812.560439
2,1,2015-12-01,2015-11-28,2462.0,2056.469009,1806.922363,2054.104514,2048.726287,2053.915391,1264.259438,2848.678580,1137.117725,2476.727002,1202.881435,2905.327593,1231.112298,2866.340276,1208.765781,2899.065000
3,1,2015-12-02,2015-11-28,2520.0,2389.871105,2279.667236,2488.622850,2482.115711,2488.398833,1453.012104,3326.730106,1672.117664,2887.216809,1457.973918,3519.271782,1474.755765,3489.475657,1461.993307,3514.804360
4,1,2015-12-03,2015-11-28,2186.0,1845.132887,1925.597046,2231.952538,2224.262236,2231.688489,1120.412122,2569.853651,1334.921069,2516.273022,1117.979623,3345.925454,1160.251798,3288.272674,1126.589383,3336.787596


# MLFlow

In [ ]:
experiment_name = "ml_forecast_multimodel"
try:
    mlflow.create_experiment(name=experiment_name)
    meta = mlflow.get_experiment_by_name(experiment_name)
    print(f"Experiment '{experiment_name}' created with ID: {meta.experiment_id}")
except:
    meta = mlflow.get_experiment_by_name(experiment_name)
    print(f"Experiment '{experiment_name}' already exists with ID: {meta.experiment_id}")

run_time = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
run_name = f"lightgbm6_{run_time}"

# Start the run
with mlflow.start_run(experiment_id=meta.experiment_id, run_name=run_name) as run:
    # Log the model
    mlforecast.flavor.log_model(model=mlf, artifact_path="prod_model")
    mlflow.log_params(params)
    print(f"MLflow Run created - Name: {run_name}, ID: {run.info.run_id}")

2026/06/11 00:30:22 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Experiment 'ml_forecast_multimodel' created with ID: 5


AttributeError: 'MLForecast' object has no attribute 'models_'